# Phase 6.5 — volatility feature comparison

This notebook tests whether the two approved volatility inputs help. It compares all 14 approved market inputs with exactly the same set **without volatility**. Return and risk are checked separately across the same four historical time folds.

It fits models. Run it only after selecting the project `.venv` kernel. It creates a new local evidence folder and will refuse to overwrite an existing one.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'src' / 'model_training').is_dir():
    root = root.parent
if not (root / 'src' / 'model_training').is_dir():
    raise FileNotFoundError('Open this notebook from inside the HalalifyAI repository.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
%cd {root}
print(f'Project folder: {root}')

## Run the controlled comparison

The next cell uses only the verified Phase 6.2 recovery dataset and its time-based protocol. It tunes only on earlier inner-validation data, then scores each later outer fold once. Expect a few minutes of work.

In [ ]:
from src.model_training.feature_ablation.ablation_runner import (
    DEFAULT_RECOVERY_VOLATILITY_OUTPUT,
    write_recovery_volatility_ablation_report,
)

report = write_recovery_volatility_ablation_report()
print('Status:', report['status'])
print('Evidence folder:', DEFAULT_RECOVERY_VOLATILITY_OUTPUT)
for task, summary in report['task_summaries'].items():
    print(f"{task}: {summary['decision']} | {summary['primary_metric']} difference = {summary['mean_difference']:.4f}")

## Read the result

`retain` means volatility consistently helped. `reject` means it did not help in most time periods. `inconclusive` means the evidence was mixed. For risk, a result cannot be retained if High-risk detection becomes worse or if more High-risk cases are wrongly predicted as Low risk.

In [ ]:
for task, summary in report['task_summaries'].items():
    print('\n', task.upper())
    display(summary['fold_metrics'])
    if task == 'risk':
        print('Risk safety checks:', summary['risk_guardrails'])